# MVO4 Swan 8: REPLICATION_GUIDE.md implementation

This revised notebook uses the shared `mvo_algo.py` pipeline. Run from the `replication` directory. The uploaded notebook is preserved separately. The guide controls all dates, swans, and model choices. No live downloads are used.

In [1]:
import os, sys
from pathlib import Path
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[name] = "1"
HERE = Path.cwd()
if not (HERE / "mvo_algo.py").exists():
    raise RuntimeError("Run this notebook from the replication directory")
sys.path.insert(0, str(HERE))
import pandas as pd
import matplotlib.pyplot as plt
from mvo_algo import Replication, EVENTS, versions, write_outputs
model = Replication(cache=HERE / ".cache/dbi")
versions()

{'python': '3.12.14',
 'numpy': '2.2.3',
 'pandas': '2.3.3',
 'scikit-learn': '1.7.2',
 'scipy': '1.16.3',
 'PyPortfolioOpt': '1.6.0',
 'cvxpy': '1.9.2'}

## 1–3. Historical inputs, plain-average crash shape, and inclusive window

In [2]:
EVENT = 'Swan 8'
K, Seeds, MinAppear, w1 = 7, 3, 3, 0.17
ev = EVENTS[EVENT]
print(ev)
print("Eligible universe:", len(model.training_beta(EVENT)))
print("t1, t2, dd:", model.curve_params(EVENT))
print("start, end, extended:", model.training_window(EVENT))

Event(name='Swan 8', peak='2018-10-03', bottom='2018-12-24', train_swans=('Swan3', 'Swan4', 'Swan5', 'Swan6', 'Swan7'), optdata='Swan8_sp100_optdata4.xlsx')
Eligible universe: 78
t1, t2, dd: (74, 216, 0.2538755845787302)
start, end, extended: (Timestamp('2017-08-07 00:00:00'), Timestamp('2018-09-28 00:00:00'), True)


## 4. Rank all 1,000 seeds, vote, retain the top 22 with at least MinAppear votes

In [3]:
ranking = model.dbi_ranking(EVENT, K)
print("Best seeds:", ranking[:Seeds])
selected, votes = model.preselect(EVENT, K, Seeds, MinAppear)
display(votes.to_frame())
display(selected)
assert len(selected) >= 5, "This combination has no_preselect status"

Best seeds: [(541, 1.1349511331448785), (880, 1.1434484261848679), (408, 1.1447074298299678)]


,votes
AMZN,3
CL,3
DUK,3
EXC,3
JNJ,3
KO,3
LLY,3
MO,3
NEE,3
PEP,3


,Swan3,Swan4,Swan5,Swan6,Swan7,avg
Ticker,,,,,,
AMZN,1.584517,-0.445606,0.328167,0.123350,0.334033,0.384892
CL,1.284790,0.099788,0.617542,-0.045657,0.784583,0.548209
DUK,-0.519440,1.580781,0.771714,-0.322793,0.233701,0.348793
EXC,-0.975571,0.519667,0.794675,0.263389,0.605246,0.241481
JNJ,0.565647,0.403161,0.739963,0.360346,0.668742,0.547572
KO,1.283611,-0.170562,0.536675,0.325968,0.614677,0.518074
LLY,0.125934,0.746268,0.920577,0.280671,0.780342,0.570758
MO,-0.274878,0.818853,0.448863,-0.137637,0.264024,0.223845
NEE,-0.420653,0.414534,0.503894,0.415367,0.192033,0.221035


## 5–6. Inject at the front, then compute last-year drawdown and weekly covariance ×51

In [4]:
risk = model.risk_inputs(EVENT, selected)
print("Trading days:", risk["total_days"], "Cutoff:", risk["cutoff"])
pd.Series(risk["curve"], index=risk["raw_prices"].index).plot(title="Injected market drawdown")
plt.show()
display(pd.DataFrame({"fall": risk["fall"], "R_fwd_30_MA": risk["R_fwd"], "E_2": risk["E_2"]}))
display(risk["cov"])

Trading days: 290 Cutoff: 2017-09-28 00:00:00


,fall,R_fwd_30_MA,E_2
Ticker,,,
AMZN,0.138982,0.097508,0.701586
CL,0.181611,0.010854,0.059765
DUK,0.153460,0.046478,0.302865
EXC,0.129074,0.039071,0.302705
JNJ,0.169778,0.045274,0.266664
KO,0.122908,0.105341,0.857071
LLY,0.149175,-0.006310,-0.042300
MO,0.248572,0.102285,0.411490
NEE,0.081769,0.035606,0.435449


Ticker,AMZN,CL,DUK,EXC,JNJ,KO,LLY,MO,NEE,PEP,PG,SBUX,SO,UNH,VZ
Ticker,,,,,,,,,,,,,,,
AMZN,0.062393,-0.004516,0.003847,0.007971,0.008915,0.010336,0.014418,-0.002369,0.009283,0.007979,0.002091,0.011619,-0.004590,0.019037,0.011264
CL,-0.004516,0.033715,0.004065,0.003737,0.016373,0.017098,0.016341,0.026272,0.005199,0.025248,0.026695,0.022949,0.003740,0.007733,0.012616
DUK,0.003847,0.004065,0.023123,0.016181,0.007045,0.005624,0.011139,0.004386,0.017148,0.008230,0.002613,-0.000302,0.018665,0.006089,0.014627
EXC,0.007971,0.003737,0.016181,0.019694,0.006174,0.006613,0.009501,0.006439,0.015760,0.006755,0.004207,0.002749,0.012167,0.004960,0.007723
JNJ,0.008915,0.016373,0.007045,0.006174,0.033067,0.020326,0.031718,0.022656,0.006947,0.017979,0.020697,0.022779,0.003186,0.024181,0.026217
KO,0.010336,0.017098,0.005624,0.006613,0.020326,0.024775,0.022348,0.021148,0.009492,0.021169,0.018798,0.018071,0.002716,0.018095,0.019480
LLY,0.014418,0.016341,0.011139,0.009501,0.031718,0.022348,0.050506,0.019864,0.008901,0.020049,0.023071,0.027718,0.004846,0.029594,0.034098
MO,-0.002369,0.026272,0.004386,0.006439,0.022656,0.021148,0.019864,0.047711,0.010264,0.027081,0.026413,0.018857,0.002518,0.012476,0.017663
NEE,0.009283,0.005199,0.017148,0.015760,0.006947,0.009492,0.008901,0.010264,0.023314,0.010935,0.004070,-0.000314,0.012680,0.007170,0.013402


## 7–8. Max Sharpe at rf=0, 3% floor, then buy at peak and evaluate at bottom

In [5]:
result = model.run_one(EVENT, K, Seeds, MinAppear, w1, detail=True)
print("Status:", result["status"], "PR:", result["PR"])
assert result["status"] == "ok", result.get("error")
display(result["weights"].to_frame("weight"))
pd.DataFrame({"portfolio": result["evaluation"]["portfolio_values"], "benchmark": result["evaluation"]["benchmark_values"]}).plot(title=EVENT)
plt.show()

Status: ok PR: 0.531485587158176


,weight
AMZN,0.175527
DUK,0.118068
EXC,0.075972
KO,0.175527
MO,0.103850
NEE,0.175527
UNH,0.175527


## 9. Small sweep and optional exports

The command-line runner handles the complete 5,376-run grid. Below is a four-combination event sweep; failed optimization is retained as a status, with no alternate model.

In [6]:
small = model.run_sweep(dict(K=[7], Seeds=[3], MinAppear=[3], w1=[0.05, 0.11, 0.15, 0.17]), [EVENT])
display(small)
SAVE_OUTPUTS = False
if SAVE_OUTPUTS:
    output = HERE / "results" / EVENT.replace(" ", "_")
    write_outputs(small, output, [EVENT])
    model.export_checkpoints(output / "replication_checkpoints.xlsx", K=K, Seeds=Seeds, MinAppear=MinAppear, w1=w1)

Swan 8, K=7: 4 runs completed


,event,K,Seeds,MinAppear,w1,status,PR,port_dd,market_dd,n_preselect,n_final,holdings,error,train_start,train_end
0,Swan 8,7,3,3,0.05,opt_failed,NaN,NaN,NaN,15,0,{},infeasible: n_stocks * w1 < 1,2017-08-07,2018-09-28
1,Swan 8,7,3,3,0.11,ok,0.724030,0.055082,0.199593,15,9,"{""AMZN"": 0.11111111111111112, ""DUK"": 0.1111111...",None,2017-08-07,2018-09-28
2,Swan 8,7,3,3,0.15,ok,0.605445,0.078750,0.199593,15,8,"{""AMZN"": 0.150996577410912, ""DUK"": 0.149889269...",None,2017-08-07,2018-09-28
3,Swan 8,7,3,3,0.17,ok,0.531486,0.093512,0.199593,15,7,"{""AMZN"": 0.1755273564547604, ""DUK"": 0.11806796...",None,2017-08-07,2018-09-28


Limitations: original checkpoint workbook is missing; fresh Yahoo downloads and alternative experimental objectives are outside this replication workflow. Swan9 analyst inputs have the guide's point-in-time caveat. See `IMPLEMENTATION_REPORT.md` for validation results.